## Gold Baseline Modeling (Deliverable 1.3.2)

This notebook implements the baseline anomaly detection model using a single, broad Isolation Forest trained on the complete Gold feature set. The baseline serves as the primary comparison point for evaluating whether the three-stage cascade model improves alert quality.

**Purpose:**  
To produce the baseline model’s anomaly scores and alert outputs using the fully preprocessed Gold dataset. These outputs represent the simplest form of unsupervised anomaly detection and form the quantitative reference for the comparative evaluation in the Gold Comparison notebook.

**Key Goals:**

- Load the Gold preprocessed dataset and Stage 1 feature set.
- Train a single Isolation Forest using all vetted numeric features.
- Generate anomaly scores and binary alert flags.
- Produce baseline alert frequency counts, false-positive counts, and normal-period alert rates.
- Save all baseline model artifacts, metrics, and alert outputs for use in the Gold Comparison notebook.

**Relevance to Section C:**  
This notebook provides the baseline alert patterns used in Section C to evaluate whether the cascade reduces false positives, reduces noisy alerts, and preserves meaningful anomaly sensitivity. The outputs here form the “reference condition” for the paired statistical tests and practical significance analysis.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

#import os
#import glob

from pathlib import Path
import yaml
import re

import logging
import wandb

import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

import joblib 

from sklearn.model_selection import train_test_split, KFold

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, RobustScaler

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score

from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

import pyarrow.parquet as pq
import pyarrow as pa


import hashlib


# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import load_data, save_data, save_json, load_json
from utils.eda_logging import profile_dataframe
from utils.logging_setup import configure_logging
from utils.wandb_utils import finalize_wandb_stage

# Ledger 
from utils.ledger import Ledger

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def log_gold_paths(paths, logger: logging.Logger) -> None:
    logger.info("Project Root Path Loaded: %s", paths.root)
    logger.info("Project Logging Path Loaded: %s", paths.logs)
    logger.info("Project Artifacts Path Loaded: %s", paths.artifacts)
    logger.info("Project Notebooks Path Loaded %s", paths.notebooks)
    logger.info("Project Data Path Loaded: %s", paths.data)
    logger.info("Data Bronze Path Loaded: %s", paths.data_bronze)
    logger.info("Data Bronze Training Path Loaded: %s", paths.data_bronze_train)
    logger.info("Data Bronze Testing Path Loaded: %s", paths.data_bronze_test)
    logger.info("Data Silver Path Loaded: %s", paths.data_silver)
    logger.info("Data Silver Training Path Loaded: %s", paths.data_silver_train)
    logger.info("Data Silver Testing Path Loaded: %s", paths.data_silver_test)
    logger.info("Data Gold Path Loaded: %s", paths.data_gold)

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Configurables

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Stage Details
STAGE = "gold"
LAYER_NAME = "gold"
GOLD_VERSION = "gold__001"
RECIPE_ID = "gold_modeling__v001_baseline"


DATASET_NAME_CONFIG = "pump"
DATASET_NAME = str(DATASET_NAME_CONFIG).strip().lower()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Weights and Biases
WANDB_PROJECT = "capstone"
WANDB_ENTITY = "dcoo230-western-governors-university"
WANDB_RUN_NAME = f"{GOLD_VERSION}"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# File names
GOLD_FILE_NAME = f"{DATASET_NAME}__gold__preprocessed.parquet"
STAGE1_FEATURES_FILE_NAME = f"{DATASET_NAME}__gold__stage1_features.json"


#BASELINE_RESULTS_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_results.csv"
#BASELINE_MODEL_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_isolation_forest.joblib"
#BASELINE_THRESHOLDS_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_thresholds.json"
#BASELINE_SUMMARY_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_summary.json"
#BASELINE_METADATA_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_metadata.json"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Split configuration
TRAIN_FRACTION = 0.70
RANDOM_SEED = 42

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Scaling posture
USE_ROBUST_SCALER = False

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Baseline thresholding
BASELINE_THRESHOLD_PERCENTILE = 97.0

# Cascade thresholding
STAGE1_THRESHOLD_PERCENTILE = 95.0   # broader / more sensitive
STAGE2_THRESHOLD_PERCENTILE = 98.5   # narrower / stricter

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Isolation Forest size
BASELINE_ESTIMATOR_COUNT = 200


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 







In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Paths Setup

# Get Path's Object
paths = get_paths()

# Gold
GOLD_DATA_PATH = paths.data_gold / GOLD_FILE_NAME
GOLD_ARTIFACTS_PATH = paths.artifacts / "gold" / DATASET_NAME

# Baseline outputs
BASELINE_RESULTS_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_results.csv"
BASELINE_MODEL_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_isolation_forest.joblib"
BASELINE_THRESHOLDS_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_thresholds.json"
BASELINE_SUMMARY_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_summary.json"
BASELINE_METADATA_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_metadata.json"

STAGE1_FEATURES_PATH = GOLD_ARTIFACTS_PATH / STAGE1_FEATURES_FILE_NAME

# Logs
LOGS_PATH = paths.logs

GOLD_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)



In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_modeling_baseline.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold Modeling stage starting")

# Log paths loads
log_gold_paths(paths, logger)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_modeling_baseline",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "train_fraction": TRAIN_FRACTION,
        "baseline_threshold_percentile": BASELINE_THRESHOLD_PERCENTILE,
        "gold_input_path": str(GOLD_DATA_PATH),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
logger.info("Loading Gold parquet: %s", GOLD_DATA_PATH)
gold_working_dataframe = load_data(GOLD_DATA_PATH)

logger.info("Loading Stage 1 features: %s", STAGE1_FEATURES_PATH)
stage1_feature_columns = load_json(STAGE1_FEATURES_PATH)

ledger.add(
    kind="step",
    step="load_modeling_inputs",
    message="Loaded Gold parquet and stage feature artifacts for modeling.",
    data={
        "gold_path": str(GOLD_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "gold_shape": list(gold_working_dataframe.shape),
        "stage1_feature_count": int(len(stage1_feature_columns)),
    },
    logger=logger,
)

gold_working_dataframe.head(3)

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def build_time_ordered_split_mask(
    dataframe: pd.DataFrame,
    *,
    train_fraction: float,
) -> tuple[pd.Series, dict]:
    if "time_index" in dataframe.columns:
        ordering_series = dataframe["time_index"].astype("int64")
        ordering_column = "time_index"
    elif "event_step" in dataframe.columns:
        ordering_series = dataframe["event_step"].astype("int64")
        ordering_column = "event_step"
    else:
        ordering_series = pd.Series(np.arange(len(dataframe), dtype=np.int64), index=dataframe.index)
        ordering_column = "row_index"

    sorted_index = ordering_series.sort_values().index
    train_row_count = int(np.floor(len(sorted_index) * train_fraction))

    train_index_values = set(sorted_index[:train_row_count].tolist())
    train_mask = dataframe.index.to_series().apply(lambda index_value: index_value in train_index_values)

    split_info = {
        "train_fraction": float(train_fraction),
        "train_rows": int(train_mask.sum()),
        "test_rows": int((~train_mask).sum()),
        "ordering_column": ordering_column,
    }

    return train_mask, split_info

In [ ]:
train_mask, split_info = build_time_ordered_split_mask(
    gold_working_dataframe,
    train_fraction=TRAIN_FRACTION,
)

ledger.add(
    kind="step",
    step="build_modeling_split",
    message="Rebuilt time-ordered train/test split for Gold modeling notebook.",
    data=split_info,
    logger=logger,
)

split_info

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def get_training_rows_for_unsupervised_model(
    dataframe: pd.DataFrame,
    *,
    train_mask: pd.Series,
) -> pd.DataFrame:
    training_subset = dataframe.loc[train_mask].copy()

    if "anomaly_flag" in training_subset.columns:
        training_subset = training_subset[training_subset["anomaly_flag"] == 0].copy()

    return training_subset

In [ ]:
train_rows_for_fit = get_training_rows_for_unsupervised_model(
    gold_working_dataframe,
    train_mask=train_mask,
)

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def build_reference_profile(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> pd.DataFrame:
    reference_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]

        reference_rows.append({
            "feature_name": feature_name,
            "median_value": float(feature_series.median()),
            "mean_value": float(feature_series.mean()),
            "standard_deviation": float(feature_series.std()) if not pd.isna(feature_series.std()) else 0.0,
            "lower_bound": float(feature_series.quantile(0.05)),
            "upper_bound": float(feature_series.quantile(0.95)),
        })

    reference_profile = pd.DataFrame(reference_rows)
    return reference_profile

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
baseline_train_fit_features = train_rows_for_fit[stage1_feature_columns].values
baseline_all_features = gold_working_dataframe[stage1_feature_columns].values

if "anomaly_flag" in gold_working_dataframe.columns:
    all_labels = gold_working_dataframe["anomaly_flag"].astype(int).values
    test_labels = gold_working_dataframe.loc[~train_mask, "anomaly_flag"].astype(int).values
else:
    all_labels = None
    test_labels = None

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def compute_anomaly_scores_isolation_forest(
    model: IsolationForest,
    feature_matrix: np.ndarray,
) -> np.ndarray:
    scores = model.score_samples(feature_matrix)
    anomaly_scores = -scores
    return anomaly_scores

In [ ]:
def choose_threshold_by_percentile(
    anomaly_scores: np.ndarray,
    percentile: float,
) -> float:
    return float(np.percentile(anomaly_scores, percentile))

In [ ]:

def evaluate_against_labels(
    true_labels: np.ndarray,
    anomaly_scores: np.ndarray,
    threshold: float,
) -> dict:
    predicted_labels = (anomaly_scores >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predicted_labels,
        average="binary",
        zero_division=0,
    )

    results = {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

    if len(np.unique(true_labels)) == 2:
        results["roc_auc"] = float(roc_auc_score(true_labels, anomaly_scores))
        results["pr_auc"] = float(average_precision_score(true_labels, anomaly_scores))
    else:
        results["roc_auc"] = None
        results["pr_auc"] = None

    return results

In [ ]:

baseline_model = IsolationForest(
    n_estimators=BASELINE_ESTIMATOR_COUNT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

baseline_model.fit(baseline_train_fit_features)

baseline_train_scores = compute_anomaly_scores_isolation_forest(
    baseline_model,
    train_rows_for_fit[stage1_feature_columns].values,
)

baseline_all_scores = compute_anomaly_scores_isolation_forest(
    baseline_model,
    baseline_all_features,
)

baseline_threshold = choose_threshold_by_percentile(
    baseline_train_scores,
    BASELINE_THRESHOLD_PERCENTILE,
)

baseline_flags = (baseline_all_scores >= baseline_threshold).astype(int)

baseline_results = gold_working_dataframe.copy()
baseline_results["baseline_score"] = baseline_all_scores
baseline_results["baseline_flag"] = baseline_flags

baseline_metrics = {
    "model": "Baseline IsolationForest",
    "threshold_percentile": BASELINE_THRESHOLD_PERCENTILE,
    "threshold": float(baseline_threshold),
    "alert_count_all_rows": int(baseline_flags.sum()),
    "alert_count_test_rows": int(baseline_flags[~train_mask.values].sum()),
}

if test_labels is not None:
    baseline_metrics.update(
        evaluate_against_labels(
            test_labels,
            baseline_all_scores[~train_mask.values],
            baseline_threshold,
        )
    )

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="run_baseline_isolation_forest",
    message="Ran baseline single broad Isolation Forest across Stage 1 feature set.",
    data={
        "estimator_count": int(BASELINE_ESTIMATOR_COUNT),
        "threshold_percentile": float(BASELINE_THRESHOLD_PERCENTILE),
        "threshold": float(baseline_threshold),
        "training_rows": int(len(train_rows_for_fit)),
        "feature_count": int(len(stage1_feature_columns)),
        "alert_count_all_rows": int(baseline_metrics["alert_count_all_rows"]),
        "alert_count_test_rows": int(baseline_metrics["alert_count_test_rows"]),
        "precision": baseline_metrics.get("precision"),
        "recall": baseline_metrics.get("recall"),
        "f1": baseline_metrics.get("f1"),
        "roc_auc": baseline_metrics.get("roc_auc"),
        "pr_auc": baseline_metrics.get("pr_auc"),
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

baseline_metrics

In [ ]:
baseline_alert_count_all_rows = int(baseline_results["baseline_flag"].sum())
baseline_alert_count_test_rows = int(baseline_results.loc[~train_mask, "baseline_flag"].sum())

baseline_thresholds = {
    "baseline_threshold_percentile": float(BASELINE_THRESHOLD_PERCENTILE),
    "baseline_threshold": float(baseline_threshold),
}

baseline_summary = {
    "dataset_name": DATASET_NAME,
    "baseline_metrics": baseline_metrics,
    "alert_count_all_rows": baseline_alert_count_all_rows,
    "alert_count_test_rows": baseline_alert_count_test_rows,
    "result_row_count": int(len(baseline_results)),
}

baseline_metadata = {
    "baseline_results_path": str(BASELINE_RESULTS_PATH),
    "baseline_model_path": str(BASELINE_MODEL_PATH),
    "baseline_thresholds_path": str(BASELINE_THRESHOLDS_PATH),
    "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
}

baseline_results.to_csv(BASELINE_RESULTS_PATH, index=False)
joblib.dump(baseline_model, BASELINE_MODEL_PATH)

save_json(baseline_thresholds, BASELINE_THRESHOLDS_PATH)
save_json(baseline_summary, BASELINE_SUMMARY_PATH)
save_json(baseline_metadata, BASELINE_METADATA_PATH)

wandb.save(str(BASELINE_RESULTS_PATH))
wandb.save(str(BASELINE_MODEL_PATH))
wandb.save(str(BASELINE_THRESHOLDS_PATH))
wandb.save(str(BASELINE_SUMMARY_PATH))
wandb.save(str(BASELINE_METADATA_PATH))

ledger.add(
    kind="step",
    step="save_baseline_outputs",
    message="Saved baseline results, trained Isolation Forest model, thresholds, summary, and metadata.",
    data={
        "baseline_results_path": str(BASELINE_RESULTS_PATH),
        "baseline_model_path": str(BASELINE_MODEL_PATH),
        "baseline_thresholds_path": str(BASELINE_THRESHOLDS_PATH),
        "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
        "baseline_metadata_path": str(BASELINE_METADATA_PATH),
        "result_row_count": int(len(baseline_results)),
        "alert_count_all_rows": baseline_alert_count_all_rows,
        "alert_count_test_rows": baseline_alert_count_test_rows,
    },
    logger=logger,
)

In [ ]:
ledger.add(
    kind="step",
    step="finalize_baseline_modeling",
    message="Gold baseline modeling notebook complete.",
    data={
        "baseline_metrics": baseline_metrics,
        "baseline_results_path": str(BASELINE_RESULTS_PATH),
        "baseline_model_path": str(BASELINE_MODEL_PATH),
    },
    logger=logger,
)

baseline_ledger_path = GOLD_ARTIFACTS_PATH / f"ledger__{DATASET_NAME}__gold_baseline_modeling.json"
ledger.write_json(baseline_ledger_path)

wandb.save(str(baseline_ledger_path))
wandb_run.finish()